In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# =====================================================================
# FINAL OPTIMIZED SYSTEM
# Mistral-7B + Hybrid Logit + Generation + Calibration + Heuristics
# =====================================================================

# ================================
# INSTALL DEPENDENCIES
# ================================
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "bitsandbytes",
    "pandas", "scikit-learn", "tqdm"
], check=True)

# ================================
# IMPORTS
# ================================
import os
import gc
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ================================
# CONFIG
# ================================
DATA_PATH = "/kaggle/input/datasets/nikunjnawal009/mistral-devignx26"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
BATCH_SIZE = 4
SUBSET_SIZE = 200
MAX_LEN = 1024
THRESHOLD = 0.43

# ================================
# DATA LOADING
# ================================
def load_data(path):
    file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
    df = pd.read_csv(os.path.join(path, file))

    df = df[["code", "label"]].dropna()
    df = df[df["label"].isin([0, 1])]
    df = df.rename(columns={"code": "text"}).reset_index(drop=True)

    print("Dataset:", df.shape)
    print(df["label"].value_counts(), "\n")
    return df.head(SUBSET_SIZE)

# ================================
# MODEL LOAD
# ================================
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb,
        torch_dtype=torch.float16
    )

    model.eval()
    return model, tokenizer

# ================================
# TOKEN IDS
# ================================
def get_ids(tokenizer):
    id0 = tokenizer("0", add_special_tokens=False)["input_ids"][0]
    id1 = tokenizer("1", add_special_tokens=False)["input_ids"][0]
    return id0, id1

# ================================
# PROMPT
# ================================
PROMPT = """[INST] Classify the code:

0 = Safe
1 = Vulnerable

Rules:
- Decide only based on code
- Output ONLY 0 or 1

Example:
int a = 5;
Answer: 0

Example:
char buf[10]; gets(buf);
Answer: 1

Code:
{code}
[/INST]
Answer:"""

def prepare_code(code):
    code = code.strip()
    if len(code) > 800:
        code = code[:400] + "\n...\n" + code[-400:]
    return code

# ================================
# HYBRID PREDICTION
# ================================
def predict_batch(texts, model, tokenizer, id0, id1):
    preds = []

    for code in texts:
        code = prepare_code(code)
        prompt = PROMPT.format(code=code)

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1]

            l0 = logits[id0].item()
            l1 = logits[id1].item()

            probs = torch.softmax(torch.tensor([l0, l1]), dim=0)
            p1 = probs[1].item()

        # Generation signal
        try:
            gen = model.generate(
                **inputs,
                max_new_tokens=3,
                temperature=0.0,
                do_sample=False
            )
            out = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            gen_pred = 1 if out.strip().startswith("1") else 0
        except:
            gen_pred = 0

        # Heuristic boost
        keywords = ["gets(", "strcpy(", "memcpy(", "sprintf(", "scanf(", "malloc(", "free(", "*p = NULL"]
        if any(k in code for k in keywords):
            p1 += 0.1

        # Final score
        score = (0.6 * p1) + (0.4 * gen_pred)

        pred = 1 if score > THRESHOLD else 0
        preds.append(pred)

    return preds

# ================================
# INFERENCE
# ================================
def run(df, model, tokenizer, id0, id1):
    texts = df["text"].tolist()
    preds = []

    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[i:i+BATCH_SIZE]
        preds.extend(predict_batch(batch, model, tokenizer, id0, id1))

        torch.cuda.empty_cache()
        gc.collect()

    df["predicted_label"] = preds
    return df

# ================================
# EVALUATION
# ================================
def evaluate(df):
    y_true = df["label"]
    y_pred = df["predicted_label"]

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print("\n===== FINAL RESULTS =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}\n")

    print(classification_report(y_true, y_pred, zero_division=0))

    return acc, prec, rec, f1

# ================================
# SAVE
# ================================
def save(df, metrics):
    df.to_csv("predictions.csv", index=False)

    pd.DataFrame([{
        "accuracy": metrics[0],
        "precision": metrics[1],
        "recall": metrics[2],
        "f1_score": metrics[3]
    }]).to_csv("results.csv", index=False)

# ================================
# MAIN
# ================================
def main():
    df = load_data(DATA_PATH)
    model, tokenizer = load_model()
    id0, id1 = get_ids(tokenizer)

    df = run(df, model, tokenizer, id0, id1)
    metrics = evaluate(df)
    save(df, metrics)

    print("\nSaved predictions.csv and results.csv")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
Dataset: (2732, 2)
label
0    1545
1    1187
Name: count, dtype: int64 



config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


  0%|          | 0/50 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  2%|▏         | 1/50 [00:05<04:41,  5.74s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.

  4%|▍         | 2/50 [00:10<04:11,  5.23s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-e


===== FINAL RESULTS =====
Accuracy : 0.5250
Precision: 0.4870
Recall   : 0.8242
F1 Score : 0.6122

              precision    recall  f1-score   support

           0       0.65      0.28      0.39       109
           1       0.49      0.82      0.61        91

    accuracy                           0.53       200
   macro avg       0.57      0.55      0.50       200
weighted avg       0.58      0.53      0.49       200


Saved predictions.csv and results.csv
